# Churn Prediction — Exploratory Analysis

This notebook walks through the full pipeline:
1. Generate synthetic SaaS customer data
2. Feature engineering & time-based splits
3. Train a Logistic Regression baseline and an XGBoost model
4. Evaluate with ROC-AUC, calibration, and lift curves
5. SHAP global explainability
6. Business impact simulation


In [ ]:
import sys, pathlib
# Ensure the project root is on the path when running from the notebooks/ directory
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Generate Synthetic Dataset

In [ ]:
from src.preprocessing.pipeline import (
    generate_synthetic_dataset,
    add_engineered_features,
    time_based_split,
    generate_cohorts,
)

df_raw = generate_synthetic_dataset(n_customers=5000, seed=42)
print(f"Dataset shape: {df_raw.shape}")
print(f"Churn rate: {df_raw['churned'].mean():.2%}")
df_raw.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

numeric_cols = ['tenure_months', 'monthly_spend', 'support_tickets',
                'login_frequency', 'feature_adoption_score']

for ax, col in zip(axes.flat, numeric_cols):
    for label, grp in df_raw.groupby('churned'):
        ax.hist(grp[col], bins=30, alpha=0.6, label=f'churned={label}', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

# Churn rate by contract type
ax = axes.flat[-1]
df_raw.groupby('contract_type')['churned'].mean().plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'])
ax.set_title('Churn Rate by Contract Type')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 3. Feature Engineering & Cohort Analysis

In [ ]:
df = add_engineered_features(df_raw)
cohorts = generate_cohorts(df, cohort_col='tenure_bucket')
print("\nCohort summary:")
print(cohorts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cohorts_sorted = cohorts.sort_values('churn_rate', ascending=False)
ax.bar(cohorts_sorted['cohort'], cohorts_sorted['churn_rate'], color=sns.color_palette('muted'))
ax.set_title('Churn Rate by Tenure Cohort')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Tenure Bucket')
plt.tight_layout()
plt.show()

## 4. Build Feature Matrix & Time-Based Split

In [ ]:
from src.features.builder import build_feature_matrix, get_feature_names_out

train_df, test_df = time_based_split(df, test_months=3)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

X_train, y_train, preprocessor = build_feature_matrix(train_df)
X_test, y_test, _ = build_feature_matrix(test_df, preprocessor=preprocessor, fit=False)
feature_names = get_feature_names_out(preprocessor)

print(f"X_train shape: {X_train.shape}, positive rate: {y_train.mean():.2%}")
print(f"X_test shape:  {X_test.shape},  positive rate: {y_test.mean():.2%}")

## 5. Train Models

In [ ]:
from src.models.baseline import train_logistic_regression, predict_proba as lr_predict
from src.models.gradient_boosting import train_xgboost

lr_model = train_logistic_regression(X_train, y_train)
print("Logistic Regression trained.")

xgb_model = train_xgboost(X_train, y_train, X_test, y_test, n_estimators=300)
print("XGBoost trained.")

## 6. Evaluate Models

In [ ]:
from src.evaluation.metrics import (
    compute_roc_auc,
    compute_calibration,
    compute_lift_curve,
    simulate_retention_impact,
)

lr_probs  = lr_predict(lr_model, X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

lr_auc  = compute_roc_auc(y_test, lr_probs)
xgb_auc = compute_roc_auc(y_test, xgb_probs)
print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"XGBoost            AUC: {xgb_auc:.4f}")

In [ ]:
# Lift curve
lift_xgb = compute_lift_curve(y_test, xgb_probs)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lift_xgb['decile'], lift_xgb['lift'], color=sns.color_palette('deep'))
ax.axhline(1.0, color='red', linestyle='--', label='Baseline')
ax.set_title('XGBoost Decile Lift Chart')
ax.set_xlabel('Decile (1 = highest risk)')
ax.set_ylabel('Lift')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Business impact simulation
impact = simulate_retention_impact(xgb_probs, intervention_cost=50, clv=500, conversion_rate=0.3)
print("\n=== Retention Impact Simulation ===")
for k, v in impact.items():
    print(f"  {k:25s}: {v}")

## 7. SHAP Global Explainability

In [ ]:
try:
    import shap
    from src.explainability.shap_explainer import compute_shap_values, get_top_features

    # Use a sample for speed
    sample_idx = np.random.choice(len(X_test), size=min(300, len(X_test)), replace=False)
    X_sample = X_test[sample_idx]

    shap_vals = compute_shap_values(xgb_model, X_sample, model_type='tree')
    top_feats = get_top_features(shap_vals, list(feature_names), top_n=10)

    print("Top 10 features by mean |SHAP| value:")
    for feat, importance in top_feats.items():
        print(f"  {feat:40s}: {importance:.4f}")

    # SHAP summary plot
    shap.summary_plot(shap_vals, X_sample, feature_names=list(feature_names), show=True)

except ImportError:
    print("shap not installed — skipping SHAP analysis.")

## 8. Survival Analysis (Cox PH)

In [ ]:
try:
    from src.models.survival import train_cox_model, predict_survival_at_t

    cox_model = train_cox_model(train_df)
    survival_12 = predict_survival_at_t(cox_model, test_df.head(5), t=12)
    print("Survival probability at 12 months (first 5 test customers):")
    for i, s in enumerate(survival_12):
        print(f"  Customer {i+1}: {s:.4f}")

except ImportError:
    print("lifelines not installed — skipping survival analysis.")

## 9. Uplift Modeling (T-Learner)

In [ ]:
try:
    from src.models.uplift import generate_treatment_data, TLearnerUplift

    df_treatment = generate_treatment_data(train_df)
    treat_mask  = df_treatment['treatment'] == 1
    ctrl_mask   = df_treatment['treatment'] == 0

    X_treat, y_treat, _ = build_feature_matrix(df_treatment[treat_mask])
    X_ctrl,  y_ctrl,  _ = build_feature_matrix(
        df_treatment[ctrl_mask], preprocessor=preprocessor, fit=False
    )

    uplift_model = TLearnerUplift()
    uplift_model.fit(X_treat, y_treat, X_ctrl, y_ctrl)

    uplift_scores = uplift_model.predict_uplift(X_test)
    print(f"Uplift score stats — mean: {uplift_scores.mean():.4f}, "
          f"std: {uplift_scores.std():.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(uplift_scores, bins=40, color='steelblue', edgecolor='white')
    ax.axvline(0, color='red', linestyle='--')
    ax.set_title('Distribution of Predicted Uplift Scores')
    ax.set_xlabel('Estimated CATE (Treatment Effect)')
    plt.tight_layout()
    plt.show()

except ImportError:
    print("lightgbm not installed — skipping uplift modeling.")

## 10. Monitoring: Drift Detection

In [ ]:
from src.monitoring.drift import monitor_prediction_drift, monitor_feature_drift

# Simulate a shifted production window
rng = np.random.default_rng(0)
shifted_probs = np.clip(xgb_probs + rng.normal(0.05, 0.05, size=len(xgb_probs)), 0, 1)

drift_result = monitor_prediction_drift(xgb_probs, shifted_probs)
print("Prediction drift report:")
for k, v in drift_result.items():
    print(f"  {k:20s}: {v}")

print()
numeric_features = ['tenure_months', 'monthly_spend', 'support_tickets']
feature_drift = monitor_feature_drift(train_df, test_df, numeric_features)
print("Feature drift report:")
for feat, stats in feature_drift.items():
    print(f"  {feat}: PSI={stats['psi']:.4f}, KS p={stats['p_value']:.4f}, drift={stats['drift_detected']}")